# Implementasi Webinar: Make Sense of Data with Analytics and AI
## Studi Kasus Simulasi PT Maju Bersama

**Narasumber webinar:** Andrea Arisal — Peneliti Senior, BRIN  
**Platform:** Python (Google Colab / Jupyter Notebook)

### Tujuan
Notebook ini menerapkan konsep utama webinar melalui dataset simulasi perusahaan distributor:
1. Memahami data dan kualitasnya.
2. Membersihkan serta menggabungkan tabel.
3. Melakukan *descriptive analytics* dan *diagnostic analytics*.
4. Membangun model prediksi keterlambatan pembayaran faktur.
5. Mengevaluasi model dengan metrik yang sesuai.
6. Menyusun rekomendasi berbasis hasil analisis.

> **Catatan penting:** Seluruh data dalam notebook ini dibuat secara sintetis untuk pembelajaran, bukan data aktual PT Maju Bersama. Hasil analisis dan model tidak boleh dianggap sebagai temuan bisnis nyata.

## 1. Persiapan pustaka

Notebook menggunakan `pandas` dan `numpy` untuk pengolahan data, `matplotlib` dan `seaborn` untuk visualisasi, serta `scikit-learn` untuk pemodelan dan evaluasi.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme()

print("Pustaka berhasil dimuat.")

## 2. Membuat dataset simulasi

Kita membuat beberapa tabel yang umum dijumpai pada perusahaan distributor:
- `customers`: informasi pelanggan.
- `products`: informasi produk dan biaya pokok.
- `invoices`: faktur penjualan, tanggal, pelanggan, dan nilai faktur.
- `invoice_items`: rincian produk dalam setiap faktur.
- `payments`: pembayaran faktur.
- `operating_costs`: biaya operasional bulanan.

Tabel dipisahkan untuk meniru kondisi data bisnis yang perlu dihubungkan melalui kunci tertentu.

In [ ]:
# Tabel pelanggan
n_customers = 120
customers = pd.DataFrame({
    "customer_id": [f"C{i:03d}" for i in range(1, n_customers + 1)],
    "customer_name": [f"Pelanggan {i:03d}" for i in range(1, n_customers + 1)],
    "region": rng.choice(["Bengkulu", "Sumatera Selatan", "Lampung", "Jambi"], n_customers),
    "customer_segment": rng.choice(["Retail", "Grosir", "Institusi"], n_customers, p=[0.50, 0.35, 0.15]),
    "credit_limit": rng.choice([10_000_000, 20_000_000, 35_000_000, 50_000_000], n_customers)
})

# Tabel produk
n_products = 18
products = pd.DataFrame({
    "product_id": [f"P{i:03d}" for i in range(1, n_products + 1)],
    "product_name": [f"Produk {i:02d}" for i in range(1, n_products + 1)],
    "category": rng.choice(["Bahan Pokok", "Minuman", "Perawatan", "Perlengkapan"], n_products),
    "unit_cost": rng.integers(15_000, 250_000, n_products),
    "unit_price": rng.integers(30_000, 350_000, n_products)
})
products["unit_price"] = np.maximum(products["unit_price"], products["unit_cost"] * 1.15).round(0)

# Tabel faktur
n_invoices = 1800
invoice_dates = pd.to_datetime(
    rng.choice(pd.date_range("2024-01-01", "2025-12-31", freq="D"), n_invoices)
).sort_values()

invoices = pd.DataFrame({
    "invoice_id": [f"INV{i:05d}" for i in range(1, n_invoices + 1)],
    "customer_id": rng.choice(customers["customer_id"], n_invoices),
    "invoice_date": invoice_dates,
})
invoices["due_date"] = invoices["invoice_date"] + pd.to_timedelta(rng.choice([14, 30, 45], n_invoices), unit="D")
invoices["invoice_amount"] = rng.integers(500_000, 12_000_000, n_invoices).astype(float)

# Tingkat keterlambatan dipengaruhi secara sintetis oleh segmen dan nilai faktur.
segment_map = customers.set_index("customer_id")["customer_segment"]
segment_risk = invoices["customer_id"].map(segment_map).map({"Retail": 0.10, "Grosir": 0.20, "Institusi": 0.28}).to_numpy()
amount_risk = np.where(invoices["invoice_amount"].to_numpy() > 8_000_000, 0.10, 0.0)
late_probability = np.clip(0.08 + segment_risk + amount_risk, 0.05, 0.65)
invoices["late_30d"] = rng.binomial(1, late_probability, n_invoices)

# Waktu pembayaran: faktur terlambat cenderung dibayar lebih dari 30 hari setelah jatuh tempo.
days_after_due = np.where(
    invoices["late_30d"].eq(1),
    rng.integers(31, 100, n_invoices),
    rng.integers(-5, 31, n_invoices)
)
invoices["payment_date"] = invoices["due_date"] + pd.to_timedelta(days_after_due, unit="D")
invoices["payment_status"] = np.where(invoices["late_30d"].eq(1), "Terlambat", "Tepat waktu")

# Rincian produk per faktur: satu sampai tiga baris per faktur.
item_rows = []
for row in invoices[["invoice_id"]].itertuples(index=False):
    chosen = rng.choice(products["product_id"], size=int(rng.integers(1, 4)), replace=False)
    for pid in chosen:
        p = products.loc[products["product_id"].eq(pid)].iloc[0]
        qty = int(rng.integers(1, 15))
        item_rows.append({
            "invoice_id": row.invoice_id,
            "product_id": pid,
            "quantity": qty,
            "unit_price": float(p["unit_price"]),
            "unit_cost": float(p["unit_cost"])
        })
invoice_items = pd.DataFrame(item_rows)
invoice_items["line_revenue"] = invoice_items["quantity"] * invoice_items["unit_price"]
invoice_items["line_cost"] = invoice_items["quantity"] * invoice_items["unit_cost"]
invoice_items["gross_profit"] = invoice_items["line_revenue"] - invoice_items["line_cost"]

# Tabel pembayaran: hanya memuat informasi yang dapat diketahui setelah pembayaran.
payments = invoices[["invoice_id", "payment_date", "payment_status"]].copy()
payments["amount_paid"] = invoices["invoice_amount"]
payments["payment_id"] = [f"PAY{i:05d}" for i in range(1, len(payments) + 1)]
payments = payments[["payment_id", "invoice_id", "payment_date", "payment_status", "amount_paid"]]

# Biaya operasional bulanan sintetis
months = pd.date_range("2024-01-01", "2025-12-01", freq="MS")
operating_costs = pd.DataFrame({
    "month": months,
    "operating_cost": rng.integers(45_000_000, 85_000_000, len(months)).astype(float)
})

print("Jumlah baris per tabel:")
for name, df in {
    "customers": customers, "products": products, "invoices": invoices,
    "invoice_items": invoice_items, "payments": payments, "operating_costs": operating_costs
}.items():
    print(f"- {name}: {len(df):,} baris")

## 3. Memahami struktur dan kualitas data

Tahap ini sesuai dengan *Data Understanding* dalam CRISP-DM. Kita memeriksa contoh data, tipe kolom, nilai kosong, duplikasi, serta hubungan kunci antartabel.

In [ ]:
display(invoices.head())
display(customers.head())
display(products.head())

def quality_report(df, table_name):
    return pd.DataFrame({
        "tabel": table_name,
        "kolom": df.columns,
        "tipe_data": [str(df[col].dtype) for col in df.columns],
        "nilai_kosong": [df[col].isna().sum() for col in df.columns],
        "jumlah_unik": [df[col].nunique(dropna=True) for col in df.columns]
    })

quality = pd.concat([
    quality_report(customers, "customers"),
    quality_report(products, "products"),
    quality_report(invoices, "invoices"),
    quality_report(invoice_items, "invoice_items"),
    quality_report(payments, "payments"),
], ignore_index=True)

display(quality)
print("Duplikasi penuh pada faktur:", invoices.duplicated().sum())
print("Duplikasi invoice_id:", invoices["invoice_id"].duplicated().sum())
print("Invoice tanpa pelanggan yang cocok:", (~invoices["customer_id"].isin(customers["customer_id"])).sum())
print("Rincian tanpa faktur yang cocok:", (~invoice_items["invoice_id"].isin(invoices["invoice_id"])).sum())

### 3.1. Menambahkan masalah kualitas data untuk latihan

Data nyata dapat memiliki nilai kosong, duplikasi, dan kategori yang tidak konsisten. Untuk memperagakan penanganannya, kita membuat salinan data yang sengaja diberi beberapa masalah. Data dasar tetap disimpan agar contoh tetap dapat diulang.

In [ ]:
invoices_raw = invoices.copy()
# Simulasi masalah kualitas data
invoices_raw.loc[invoices_raw.sample(frac=0.01, random_state=SEED).index, "invoice_amount"] = np.nan
invoices_raw.loc[invoices_raw.sample(frac=0.005, random_state=SEED + 1).index, "customer_id"] = np.nan
invoices_raw = pd.concat([invoices_raw, invoices_raw.sample(5, random_state=SEED)], ignore_index=True)

print("Sebelum dibersihkan:")
print("Baris:", len(invoices_raw))
print("Duplikasi penuh:", invoices_raw.duplicated().sum())
print("Nilai invoice_amount kosong:", invoices_raw["invoice_amount"].isna().sum())
print("Nilai customer_id kosong:", invoices_raw["customer_id"].isna().sum())

## 4. Membersihkan dan menyiapkan data

Langkah pembersihan harus mengikuti konteks dan aturan bisnis. Pada contoh ini, duplikasi penuh dihapus, faktur tanpa identitas pelanggan dikeluarkan dari analisis yang memerlukan informasi pelanggan, dan nilai nominal yang hilang diisi menggunakan median sebagai contoh sederhana.

Dalam proyek nyata, metode imputasi perlu dipilih berdasarkan penyebab nilai kosong dan pengetahuan bisnis; median bukan solusi universal.

In [ ]:
invoices_clean = invoices_raw.drop_duplicates().copy()

# Menghapus baris tanpa kunci pelanggan untuk analisis yang menggabungkan data pelanggan.
invoices_clean = invoices_clean.dropna(subset=["customer_id"])

# Contoh imputasi nominal dengan median.
median_amount = invoices_clean["invoice_amount"].median()
invoices_clean["invoice_amount"] = invoices_clean["invoice_amount"].fillna(median_amount)

# Memastikan tipe tanggal benar.
invoices_clean["invoice_date"] = pd.to_datetime(invoices_clean["invoice_date"], errors="coerce")
invoices_clean["due_date"] = pd.to_datetime(invoices_clean["due_date"], errors="coerce")
invoices_clean["payment_date"] = pd.to_datetime(invoices_clean["payment_date"], errors="coerce")

# Memeriksa aturan bisnis dasar.
invalid_dates = invoices_clean[invoices_clean["due_date"] < invoices_clean["invoice_date"]]
invalid_amounts = invoices_clean[invoices_clean["invoice_amount"] < 0]

print("Setelah dibersihkan:")
print("Baris:", len(invoices_clean))
print("Duplikasi penuh:", invoices_clean.duplicated().sum())
print("Nilai nominal kosong:", invoices_clean["invoice_amount"].isna().sum())
print("Tanggal jatuh tempo sebelum tanggal faktur:", len(invalid_dates))
print("Nominal faktur negatif:", len(invalid_amounts))

## 5. Menggabungkan tabel dengan benar

Tabel faktur memiliki satu baris per faktur, sedangkan `invoice_items` memiliki beberapa baris per faktur. Jika keduanya digabung, faktur akan muncul berulang sesuai jumlah produk. Karena itu, jangan menjumlahkan `invoice_amount` setelah menggabungkan langsung dengan rincian produk tanpa memperhatikan tingkat detail (*granularity*).

Untuk analisis produk, kita mengagregasi rincian pada tingkat produk. Untuk analisis pelanggan dan faktur, kita mempertahankan tingkat satu baris per faktur.

In [ ]:
# Agregasi penjualan dan laba kotor per produk
product_performance = (
    invoice_items
    .merge(products[["product_id", "product_name", "category"]], on="product_id", how="left", validate="many_to_one")
    .groupby(["product_id", "product_name", "category"], as_index=False)
    .agg(
        total_units=("quantity", "sum"),
        revenue=("line_revenue", "sum"),
        cost=("line_cost", "sum"),
        gross_profit=("gross_profit", "sum")
    )
    .sort_values("gross_profit", ascending=False)
)
product_performance["gross_margin_pct"] = (
    product_performance["gross_profit"] / product_performance["revenue"] * 100
).round(2)

display(product_performance.head(10))

# Dataset faktur satu baris per invoice
invoice_analysis = (
    invoices_clean
    .merge(customers, on="customer_id", how="left", validate="many_to_one")
)
print("Baris faktur bersih:", len(invoices_clean))
print("Baris setelah join pelanggan:", len(invoice_analysis))
print("invoice_id unik setelah join:", invoice_analysis["invoice_id"].nunique())

## 6. Descriptive Analytics — Apa yang terjadi?

Kita menghitung pendapatan, laba kotor berdasarkan rincian produk, biaya operasional, dan piutang simulasi. Pada notebook ini, *piutang simulasi* dihitung sebagai nominal faktur yang belum dibayar pada tanggal batas analisis. Karena data pembayaran sintetis dibuat untuk latihan, angka ini hanya menunjukkan cara menghitung indikator, bukan laporan akuntansi sebenarnya.

In [ ]:
# Agregat penjualan per bulan
invoice_analysis["month"] = invoice_analysis["invoice_date"].dt.to_period("M").dt.to_timestamp()
monthly_revenue = (
    invoice_analysis.groupby("month", as_index=False)
    .agg(revenue=("invoice_amount", "sum"), invoice_count=("invoice_id", "nunique"))
)

# Laba kotor dari rincian item dan biaya operasional
item_with_date = invoice_items.merge(
    invoices[["invoice_id", "invoice_date"]], on="invoice_id", how="left", validate="many_to_one"
)
item_with_date["month"] = item_with_date["invoice_date"].dt.to_period("M").dt.to_timestamp()
monthly_profit = item_with_date.groupby("month", as_index=False).agg(gross_profit=("gross_profit", "sum"))

monthly = (
    monthly_revenue
    .merge(monthly_profit, on="month", how="left")
    .merge(operating_costs, on="month", how="left")
)
monthly["estimated_operating_result"] = monthly["gross_profit"] - monthly["operating_cost"]

# Piutang simulasi pada tanggal batas
as_of_date = pd.Timestamp("2026-01-31")
unpaid_mask = invoices["payment_date"] > as_of_date
simulated_receivables = invoices.loc[unpaid_mask, "invoice_amount"].sum()

display(monthly.head())
print(f"Total pendapatan faktur: Rp{invoices['invoice_amount'].sum():,.0f}")
print(f"Total laba kotor dari rincian item: Rp{invoice_items['gross_profit'].sum():,.0f}")
print(f"Total biaya operasional simulasi: Rp{operating_costs['operating_cost'].sum():,.0f}")
print(f"Piutang simulasi per {as_of_date.date()}: Rp{simulated_receivables:,.0f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(monthly["month"], monthly["revenue"], marker="o", label="Pendapatan faktur")
axes[0].set_title("Pendapatan Faktur per Bulan")
axes[0].set_ylabel("Rupiah")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=45)

axes[1].plot(monthly["month"], monthly["gross_profit"], marker="o", label="Laba kotor")
axes[1].plot(monthly["month"], monthly["operating_cost"], marker="o", label="Biaya operasional")
axes[1].set_title("Laba Kotor dan Biaya Operasional per Bulan")
axes[1].set_ylabel("Rupiah")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 7. Diagnostic Analytics — Mengapa kondisi terjadi?

Analisis diagnostik membandingkan kelompok dan mencari pola yang berkaitan dengan hasil. Perbandingan ini dapat membantu menentukan area yang perlu diperiksa, tetapi **tidak membuktikan sebab-akibat** dengan sendirinya.

In [ ]:
# Keterlambatan menurut segmen pelanggan
late_by_segment = (
    invoice_analysis.groupby("customer_segment", as_index=False)
    .agg(
        total_invoices=("invoice_id", "nunique"),
        late_invoices=("late_30d", "sum"),
        avg_invoice_amount=("invoice_amount", "mean")
    )
)
late_by_segment["late_rate_pct"] = (
    late_by_segment["late_invoices"] / late_by_segment["total_invoices"] * 100
).round(2)
display(late_by_segment.sort_values("late_rate_pct", ascending=False))

# Produk dengan laba kotor tertinggi
display(product_performance[[
    "product_name", "category", "total_units", "revenue", "gross_profit", "gross_margin_pct"
]].head(10))

plt.figure(figsize=(9, 5))
sns.barplot(data=late_by_segment, x="customer_segment", y="late_rate_pct")
plt.title("Persentase Faktur Terlambat menurut Segmen Pelanggan")
plt.xlabel("Segmen pelanggan")
plt.ylabel("Faktur terlambat (%)")
plt.show()

## 8. Predictive Analytics — Memprediksi keterlambatan pembayaran

### Definisi target
Target `late_30d` bernilai 1 jika faktur dibayar lebih dari 30 hari setelah jatuh tempo, dan 0 jika tidak. Untuk demonstrasi prediksi yang realistis, fitur dibatasi pada informasi yang diasumsikan tersedia ketika faktur dibuat: nilai faktur, segmen pelanggan, wilayah, batas kredit, serta tanggal dan bulan faktur.

Kolom `payment_date`, `payment_status`, `late_30d`, dan informasi lain yang baru diketahui setelah pembayaran **tidak boleh menjadi fitur**, karena dapat menyebabkan *data leakage*.

### Batasan validasi
Data dibagi berdasarkan waktu: faktur yang lebih awal menjadi data latih dan faktur yang lebih baru menjadi data uji. Ini lebih sesuai untuk simulasi prediksi masa depan daripada pembagian acak semata.

In [ ]:
# Membuat dataset model dari data faktur bersih dan informasi pelanggan.
model_df = invoices_clean.merge(
    customers[["customer_id", "region", "customer_segment", "credit_limit"]],
    on="customer_id", how="left", validate="many_to_one"
).copy()

model_df["invoice_month"] = model_df["invoice_date"].dt.month
model_df["invoice_year"] = model_df["invoice_date"].dt.year

feature_cols = [
    "invoice_amount", "region", "customer_segment", "credit_limit",
    "invoice_month", "invoice_year"
]
target_col = "late_30d"

model_df = model_df.dropna(subset=[target_col, "invoice_date"])
model_df = model_df.sort_values("invoice_date").reset_index(drop=True)

cutoff = model_df["invoice_date"].quantile(0.80)
train_df = model_df[model_df["invoice_date"] < cutoff].copy()
test_df = model_df[model_df["invoice_date"] >= cutoff].copy()

X_train, y_train = train_df[feature_cols], train_df[target_col].astype(int)
X_test, y_test = test_df[feature_cols], test_df[target_col].astype(int)

print("Batas waktu pemisahan:", cutoff.date())
print("Data latih:", len(train_df), "| proporsi terlambat:", round(y_train.mean(), 3))
print("Data uji:", len(test_df), "| proporsi terlambat:", round(y_test.mean(), 3))

In [ ]:
numeric_features = ["invoice_amount", "credit_limit", "invoice_month", "invoice_year"]
categorical_features = ["region", "customer_segment"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=250, max_depth=8, class_weight="balanced",
        random_state=SEED
    )
}

results = []
fitted_models = {}

for name, estimator in models.items():
    pipeline = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", estimator)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1": f1_score(y_test, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_test, probabilities) if y_test.nunique() == 2 else np.nan
    })
    fitted_models[name] = pipeline

results_df = pd.DataFrame(results).sort_values("recall", ascending=False)
display(results_df.round(3))

## 9. Evaluasi model

Akurasi saja tidak cukup, terutama jika kasus keterlambatan lebih sedikit daripada kasus tepat waktu:
- **Accuracy:** proporsi seluruh prediksi yang benar.
- **Precision:** proporsi prediksi terlambat yang benar-benar terlambat.
- **Recall:** proporsi kasus terlambat yang berhasil ditemukan model.
- **F1-score:** keseimbangan antara precision dan recall.
- **ROC-AUC:** kemampuan model membedakan kelas pada berbagai ambang probabilitas.

Recall penting jika tujuan utama adalah mengurangi kasus terlambat yang terlewat. Namun, recall tinggi bisa menghasilkan lebih banyak peringatan palsu, sehingga perlu dipertimbangkan bersama precision dan biaya tindak lanjut.

In [ ]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
best_predictions = best_model.predict(X_test)

print("Model yang dipilih untuk pemeriksaan awal:", best_name)
print("\nLaporan klasifikasi:")
print(classification_report(y_test, best_predictions, target_names=["Tidak terlambat", "Terlambat"], zero_division=0))

cm = confusion_matrix(y_test, best_predictions, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Prediksi tidak terlambat", "Prediksi terlambat"],
    yticklabels=["Aktual tidak terlambat", "Aktual terlambat"]
)
plt.title(f"Confusion Matrix — {best_name}")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.tight_layout()
plt.show()

### Interpretasi evaluasi

Jalankan sel evaluasi dan lihat nilai *recall* kelas **Terlambat**. Jika recall rendah atau 0%, model belum mampu menemukan kasus keterlambatan yang menjadi fokus bisnis, meskipun akurasi keseluruhan terlihat baik.

Nilai yang diperoleh pada notebook ini berasal dari data sintetis dan dapat berbeda jika parameter atau proses pembuatan data diubah. Model harus diuji lebih lanjut, dibandingkan dengan baseline, dan dievaluasi pada data yang lebih representatif sebelum digunakan dalam praktik.

## 10. Prescriptive Analytics — Menyusun rekomendasi

Analitik preskriptif menerjemahkan temuan menjadi tindakan yang dapat dipertimbangkan. Rekomendasi di bawah ini merupakan contoh berbasis hasil simulasi, bukan keputusan otomatis.

In [ ]:
# Contoh ringkasan temuan yang dapat membantu diskusi rekomendasi
late_rate = invoices["late_30d"].mean() * 100
top_product = product_performance.iloc[0]
highest_late_segment = late_by_segment.sort_values("late_rate_pct", ascending=False).iloc[0]

print(f"1. Tingkat keterlambatan pada dataset simulasi: {late_rate:.2f}%.")
print(
    f"2. Segmen dengan tingkat keterlambatan tertinggi pada simulasi: "
    f"{highest_late_segment['customer_segment']} "
    f"({highest_late_segment['late_rate_pct']:.2f}%)."
)
print(
    f"3. Produk dengan laba kotor terbesar pada simulasi: {top_product['product_name']} "
    f"(Rp{top_product['gross_profit']:,.0f})."
)
print("\nRekomendasi awal:")
print("- Prioritaskan pemeriksaan faktur bernilai besar dan pelanggan dengan riwayat keterlambatan.")
print("- Tinjau kebijakan kredit dan proses pengingat pembayaran berdasarkan hasil validasi data.")
print("- Periksa biaya operasional dan margin produk sebelum memutuskan perubahan strategi.")
print("- Jangan menerapkan model prediksi jika recall, precision, dan validasi waktu belum memenuhi kebutuhan.")

## 11. Peran Generative AI dalam workflow

Generative AI dapat membantu menjelaskan kode, menyarankan cara memeriksa nilai kosong, atau membantu menafsirkan metrik evaluasi. Namun, pengguna harus:
- Tidak mengirimkan data pribadi atau rahasia ke layanan AI tanpa izin dan perlindungan yang sesuai.
- Memeriksa ulang kode, asumsi, dan interpretasi AI.
- Mendokumentasikan perubahan yang dibantu AI.
- Memastikan keputusan akhir didukung data dan pengetahuan bisnis.

Contoh prompt untuk bantuan analisis:

> “Jelaskan mengapa model klasifikasi dapat memiliki accuracy tinggi tetapi recall kelas keterlambatan rendah. Berikan tiga pemeriksaan yang dapat dilakukan tanpa menggunakan fitur yang baru diketahui setelah pembayaran.”

## 12. Kesimpulan

Implementasi ini menunjukkan alur analisis data dari pemahaman data hingga rekomendasi. Data dibersihkan dan digabungkan dengan memperhatikan kualitas serta *granularity*. Analisis deskriptif menjelaskan kondisi data, analisis diagnostik membandingkan kelompok, dan *machine learning* digunakan untuk memperkirakan risiko keterlambatan pembayaran.

Evaluasi model perlu mempertimbangkan lebih dari akurasi, terutama *recall* untuk kelas yang menjadi fokus. KNIME dan Generative AI dapat membantu proses analisis, tetapi validasi manusia, perlindungan data, serta interpretasi yang bertanggung jawab tetap diperlukan.

### Keterbatasan
- Dataset dibuat secara sintetis sehingga pola tidak merepresentasikan bisnis nyata.
- Indikator laba kotor dan biaya operasional disimulasikan secara terpisah; ini bukan laporan laba-rugi resmi.
- Model belum melalui optimasi ambang prediksi, validasi silang berbasis waktu, atau pengujian pada data eksternal.
- Rekomendasi harus diperlakukan sebagai latihan analitik, bukan instruksi bisnis langsung.

### Referensi konsep
- CRISP-DM: Business Understanding, Data Understanding, Data Preparation, Modeling, Evaluation, Deployment.
- Empat tingkatan analitik: descriptive, diagnostic, predictive, dan prescriptive analytics.
- Prinsip kualitas data: *garbage in, garbage out*.
- Evaluasi klasifikasi: accuracy, precision, recall, F1-score, ROC-AUC, dan confusion matrix.